# Additional file 3 (BMC Med Inform Decis Mak)
Joins the aggregated result tables of the source notebooks into one Excel file: `/kaggle/working/Additional_file_3.xlsx` (README sheet with the description and source of each sheet). CPU, Internet OFF.

Inputs (notebook outputs): benchmark-segmentacion-cv-nino, f1-iou05, map-segmentadores, iou-caja-segmentadores, dice-yolo-crudo, dice-sin-terceros-molares, benchmark-icdas-v4, analisis-icdas-v4, sensibilidad-relectura-v4, pipeline-icdas-v4. Row-level files (per photograph, crop or prediction) and internal checks are not included.

In [ ]:
import glob, os, re
import numpy as np, pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter
IN, OUT = '/kaggle/input', '/kaggle/working/Additional_file_3.xlsx'

MODEL = {'unetpp_r50': 'U-Net++ R50', 'deeplabv3p_r50': 'DeepLabV3+ R50', 'segformer_b2': 'SegFormer-B2', 'yolo11m_seg': 'YOLO11m-seg',
         'U-Net++': 'U-Net++ R50', 'DeepLabV3+': 'DeepLabV3+ R50',
         'resnet50': 'ResNet-50', 'convnextv2_tiny': 'ConvNeXt V2-T', 'vit_small': 'ViT-S/16', 'swin_tiny': 'Swin-T',
         'coatnet_0': 'CoAtNet-0', 'maxvit_tiny': 'MaxViT-T', 'yolo11m_cls': 'YOLO11-cls'}
VAL = {'grupo dentario': 'tooth group', 'vista': 'view', 'retractor': 'retractor use', 'incisivos': 'incisors', 'caninos': 'canines',
       'premolares': 'premolars', 'molares 1.º–2.º': 'first and second molars', 'terceros molares': 'third molars',
       'principal': 'main', 'no_retractors': 'without retractor', 'retractors': 'with retractor',
       'M (manual)': 'M (manual crop)', 'G (máscara de referencia)': 'G (reference-mask crop)', 'P-FDI (encontrados)': 'P-FDI (found)',
       'P-FDI extremo a extremo': 'P-FDI (end to end)', 'P-ubic.': 'P-location',
       'pintado': 'YOLO11m-seg painted', 'crudo': 'YOLO11m-seg raw', 'V1_union': 'YOLO11m-seg V1 union', 'V2_sin_umbral': 'YOLO11m-seg V2 no threshold',
       'YOLO pintado': 'YOLO11m-seg painted', 'YOLO crudo': 'YOLO11m-seg raw', 'promedio 4 clases': 'mean of 4 classes'}
TOK = {'modelo': 'model', 'ninos': 'children', 'nino': 'child', 'fotos': 'photographs', 'foto': 'photograph', 'dientes': 'teeth',
       'diente': 'tooth', 'grupo': 'group', 'nivel': 'level', 'escenario': 'scenario', 'metrica': 'metric', 'dif': 'diff',
       'lo': 'ci_low', 'hi': 'ci_high', 'media': 'mean', 'mediana': 'median', 'de': 'sd', 'enc': 'found', 'encontrados': 'found',
       'razon': 'ratio', 'area': 'area', 'mayor': 'above', 'menor': 'below', 'con': 'with', 'mas': 'more', 'componente': 'component',
       'orden': 'rank', 'cambia': 'changed', 'tipo': 'type', 'criterio': 'criterion', 'mejor': 'best', 'sens': 'sens',
       'numeracion': 'numbering', 'cubiertos': 'covered', 'franja': 'band', 'pred': 'pred', 'ref': 'ref', 'clases': 'classes',
       'umbral': 'threshold', 'acierto': 'correct', 'centro': 'central_fraction', 'por': 'per', 'recorte': 'crop', 'recortes': 'crops',
       'encontrado': 'found', 'todos': 'all', 'pct': 'prop', 'instancias': 'instances', 'valor': 'value', 'comparacion': 'comparison',
       'diferencia': 'difference', 'significativa': 'significant', 'acuerdo': 'agreement', 'exacto': 'exact', 'lineal': 'linear',
       'epoca': 'epoch', 'entrenamiento': 'training', 'tabla': 'table', 'duplicados': 'duplicated', 'pares': 'pairs', 'con': 'with', 'condicion': 'input', 'significativo': 'significant'}
FULL = {'pct_con_mas_de_1_componente': 'prop_more_than_1_component', 'fold_de': 'fold_sd', 'Unnamed: 1': 'item'}
def en_col(c):
    if c in FULL: return FULL[c]
    return '_'.join(TOK.get(t, t) for t in str(c).split('_'))
def tidy(d):
    d = d.copy()
    d.columns = [en_col(c) for c in d.columns]
    for c in d.columns:
        if d[c].dtype == object or pd.api.types.is_string_dtype(d[c]):
            d[c] = d[c].map(lambda x: MODEL.get(x, VAL.get(x, x)) if isinstance(x, str) else x)
    return d

def find(rel):
    h = sorted(glob.glob(f'{IN}/**/{rel}', recursive=True)); return h[0] if h else None

# (sheet, section, description, source notebook, file pattern relative to the notebook output)
S1, S2, CH = 'Stage 1 (segmentation and FDI numbering)', 'Stage 2 (Merged-ICDAS classification)', 'Chained pipeline'
SPEC = [
 ('S1_metrics_CI', S1, 'Per-tooth Dice and IoU, tooth-vs-background Dice, identification (Dice >= 0.5, benchmark definition), numbering accuracy, with 95% CI (child bootstrap, 1000 resamples).', 'benchmark-segmentacion-cv-nino', 'results/metricas_ic95.csv'),
 ('S1_by_fold', S1, 'Same metrics per cross-validation fold.', 'benchmark-segmentacion-cv-nino', 'results/metricas_por_fold.csv'),
 ('S1_by_tooth', S1, 'Per-tooth (FDI) Dice and identification rate, with number of reference teeth.', 'benchmark-segmentacion-cv-nino', 'results/metricas_por_diente.csv'),
 ('S1_subgroups', S1, 'Metrics by view and by retractor use.', 'benchmark-segmentacion-cv-nino', 'results/metricas_subgrupos.csv'),
 ('S1_Dice_paired', S1, 'Per-tooth Dice: best model vs each other model, paired child bootstrap with Holm adjustment.', 'benchmark-segmentacion-cv-nino', 'results/comparacion_pareada.csv'),
 ('S1_ident_IoU05', S1, 'FDI identification with IoU > 0.5 (main criterion): sensitivity, PPV and F1 with 95% CI and per-fold mean and SD.', 'f1-iou05', 'f1/f1_iou05_resumen.csv'),
 ('S1_ident_criteria', S1, 'Identification F1 under three matching criteria (Dice >= 0.5, IoU > 0.5, IoU > 0.75) and model ranks.', 'f1-iou05', 'f1/f1_umbral_sensibilidad.csv'),
 ('S1_ident_paired', S1, 'Identification F1: best model vs each other model under each criterion, paired child bootstrap with Holm adjustment.', 'f1-iou05', 'f1/f1_pareada_todos_criterios.csv'),
 ('S1_ident_by_fold', S1, 'Identification sensitivity, PPV and F1 per fold and criterion.', 'f1-iou05', 'f1/f1_por_fold.csv'),
 ('S1_ident_by_tooth', S1, 'Identification sensitivity and PPV per FDI tooth (IoU > 0.5 and Dice >= 0.5).', 'f1-iou05', 'f1/f1_iou05_por_diente.csv'),
 ('S1_ident_subgroups', S1, 'Identification by view and retractor use (IoU > 0.5 and Dice >= 0.5).', 'f1-iou05', 'f1/f1_iou05_subgrupos.csv'),
 ('S1_ident_band', S1, 'Teeth identified only under the looser criterion (1/3 <= IoU <= 0.5), by tooth group and tooth.', 'f1-iou05', 'f1/franja_033_05.csv'),
 ('S1_numbering', S1, 'Numbering accuracy among covered reference teeth (does not depend on the matching criterion).', 'f1-iou05', 'f1/numeracion_control.csv'),
 ('S1_mAP_summary', S1, 'Mask and box mAP50, mAP75 and mAP50-95 (COCO evaluator) with 95% CI (child bootstrap, 2000 resamples) and per-fold mean and SD.', 'map-segmentadores', 'map/map_resumen.csv'),
 ('S1_mAP_by_fold', S1, 'Mask and box mAP per fold.', 'map-segmentadores', 'map/map_por_fold.csv'),
 ('S1_mAP_paired', S1, 'mAP: all pairs of models, paired child bootstrap with Holm adjustment.', 'map-segmentadores', 'map/comparacion_pareada_map.csv'),
 ('S1_AP_by_tooth', S1, 'Mask and box AP per FDI tooth.', 'map-segmentadores', 'map/ap_por_diente.csv'),
 ('S1_mAP_instance_rule', S1, 'Sensitivity of mask mAP to the instance rule (main; S1 maximum score; S2 connected components; S3 YOLO11m-seg with the semantic rule) and ranks.', 'map-segmentadores', 'map/map_sensibilidad.csv'),
 ('S1_mAP_instances', S1, 'Number of reference and predicted instances per model and scenario.', 'map-segmentadores', 'map/conteo_instancias.csv'),
 ('S1_boxIoU_summary', S1, 'Box IoU per reference tooth (tooth not found = 0): mean, mean and median among found teeth, proportions >= 0.5/0.75/0.9, with 95% CI (child bootstrap, 2000 resamples) and per-fold mean and SD.', 'iou-caja-segmentadores', 'iou_caja/iou_caja_resumen.csv'),
 ('S1_boxIoU_by_fold', S1, 'Box IoU per fold.', 'iou-caja-segmentadores', 'iou_caja/iou_caja_por_fold.csv'),
 ('S1_boxIoU_paired', S1, 'Box IoU (mean and proportion >= 0.75): all pairs of models, paired child bootstrap with Holm adjustment.', 'iou-caja-segmentadores', 'iou_caja/iou_caja_pareada.csv'),
 ('S1_boxIoU_subgroups', S1, 'Box IoU by tooth group, view and retractor use.', 'iou-caja-segmentadores', 'iou_caja/iou_caja_subgrupos.csv'),
 ('S1_boxIoU_size', S1, 'Box size error: predicted / reference box area among found teeth (median, P10-P90, proportions), main rule and largest connected component.', 'iou-caja-segmentadores', 'iou_caja/iou_caja_tamano.csv'),
 ('S1_boxIoU_sensitivity', S1, 'Box IoU sensitivity analyses: largest connected component (S1), raw YOLO11m-seg instance with confidence >= 0.25 (S2) and without threshold (S2b), with ranks.', 'iou-caja-segmentadores', 'iou_caja/iou_caja_sensibilidad.csv'),
 ('S1_Dice_raw_YOLO', S1, 'Sensitivity of YOLO11m-seg to the conversion of instances into a semantic mask: per-tooth Dice and IoU, identification F1 (Dice >= 0.5 and IoU > 0.5) and HD95 for the painted mask (benchmark), the raw highest-confidence instance (confidence >= 0.25), the union of instances (V1) and no confidence threshold (V2), with SegFormer-B2 as reference; 95% CI (child bootstrap, 2000 resamples) and per-fold mean and SD.', 'dice-yolo-crudo', 'dice_yolo_crudo/dice_yolo_crudo_resumen.csv'),
 ('S1_Dice_raw_by_fold', S1, 'Same as S1_Dice_raw_YOLO, per fold.', 'dice-yolo-crudo', 'dice_yolo_crudo/dice_yolo_crudo_por_fold.csv'),
 ('S1_Dice_raw_paired', S1, 'Raw minus painted YOLO11m-seg, and SegFormer-B2 minus YOLO11m-seg (raw and painted): paired child bootstrap with Holm adjustment.', 'dice-yolo-crudo', 'dice_yolo_crudo/dice_yolo_crudo_pareada.csv'),
 ('S1_Dice_raw_subgroups', S1, 'Per-tooth Dice of the YOLO11m-seg scenarios and SegFormer-B2 by tooth group, view and retractor use.', 'dice-yolo-crudo', 'dice_yolo_crudo/dice_yolo_crudo_subgrupos.csv'),
 ('S1_Dice_no3M_summary', S1, 'Per-tooth Dice without third molars (mean over 28 FDI classes; main), over 32 classes (benchmark), weighted by tooth (with and without third molars) and third molars only, for the four models and raw YOLO11m-seg; 95% CI (child bootstrap, 2000 resamples) and per-fold mean and SD.', 'dice-sin-terceros-molares', 'dice_sin_3M/dice_sin_3M_resumen.csv'),
 ('S1_Dice_no3M_by_fold', S1, 'Same as S1_Dice_no3M_summary, per fold.', 'dice-sin-terceros-molares', 'dice_sin_3M/dice_sin_3M_por_fold.csv'),
 ('S1_Dice_no3M_paired', S1, 'Per-tooth Dice without third molars (and over 32 classes, and weighted by tooth): the six pairs of the four models plus SegFormer-B2 minus raw YOLO11m-seg, paired child bootstrap with Holm adjustment.', 'dice-sin-terceros-molares', 'dice_sin_3M/dice_sin_3M_pareada.csv'),
 ('S1_Dice_third_molars', S1, 'Dice of each third-molar class (FDI 18, 28, 38, 48) and their mean, with 95% CI, number of reference teeth, photographs and children.', 'dice-sin-terceros-molares', 'dice_sin_3M/dice_3M_solo.csv'),
 ('S2_metrics_CI', S2, 'All classification metrics on pooled out-of-fold predictions with 95% CI (child bootstrap).', 'benchmark-icdas-v4', 'results/metricas_oof_ic95.csv'),
 ('S2_by_fold', S2, 'Classification metrics per fold.', 'benchmark-icdas-v4', 'results/metricas_por_fold.csv'),
 ('S2_by_class', S2, 'Per-class precision, recall and F1 (Merged-ICDAS 0, A, B, C).', 'benchmark-icdas-v4', 'results/tabla_por_clase.csv'),
 ('S2_operating_points', S2, 'Screening cut-offs (any caries, B or C, C): sensitivity, specificity, predictive values and AUROC.', 'benchmark-icdas-v4', 'results/metricas_cortes.csv'),
 ('S2_subgroups', S2, 'Classification metrics by view and retractor use.', 'benchmark-icdas-v4', 'results/metricas_subgrupos.csv'),
 ('S2_paired', S2, 'Macro F1: best model (Swin-T) vs each other model, paired child bootstrap with Holm adjustment.', 'benchmark-icdas-v4', 'results/comparacion_pareada.csv'),
 ('S2_errors_by_ICDAS', S2, 'Predicted Merged-ICDAS class by original ICDAS code (0-6), per model.', 'benchmark-icdas-v4', 'results/errores_por_icdas.csv'),
 ('S2_confusion_ICDAS', S2, 'Confusion of original ICDAS codes vs predicted Merged-ICDAS class, all models stacked.', 'benchmark-icdas-v4', 'results/matriz_icdas_*.csv'),
 ('S2_validation', S2, 'Mean validation macro F1 (used to choose the pipeline classifier) and out-of-fold test macro F1.', 'analisis-icdas-v4', 'seleccion_por_validacion.csv'),
 ('S2_efficiency', S2, 'Parameters, GFLOPs and CPU latency (one thread).', 'analisis-icdas-v4', 'eficiencia.csv'),
 ('S2_GradCAM_central', S2, 'Grad-CAM concentration: mean fraction of the map within the central 60% of the crop, overall and by correct/incorrect prediction (300 crops per model).', 'analisis-icdas-v4', 'gradcam_centralidad.csv'),
 ('S2_relabel_sensitivity', S2, 'Sensitivity analysis of model-guided re-reading: metrics with v4 labels, original labels and excluding re-read crops.', 'sensibilidad-relectura-v4', 'sensibilidad_relectura_por_modelo.csv'),
 ('S2_relabel_paired', S2, 'Re-reading sensitivity: paired comparisons with Holm adjustment in each scenario.', 'sensibilidad-relectura-v4', 'sensibilidad_relectura_holm.csv'),
 ('S2_relabel_agreement', S2, 'Agreement between original and re-read labels (including control crops).', 'sensibilidad-relectura-v4', 'relectura_concordancia.csv'),
 ('CH_metrics', CH, 'Chained evaluation (SegFormer-B2 + Swin-T, 908 teeth): metrics for manual crops (M), reference-mask crops (G), predicted crops (P-FDI found and end to end, P-location).', 'pipeline-icdas-v4', 'pipeline_metricas.csv'),
 ('CH_detection_by_class', CH, 'Teeth found with their FDI number, by Merged-ICDAS class.', 'pipeline-icdas-v4', 'pipeline_deteccion_por_clase.csv'),
 ('CH_found_IoU05', CH, 'Teeth found with their FDI number when IoU > 0.5 (or Dice >= 0.5) with the reference tooth mask is required, overall and by class.', 'f1-iou05', 'f1/encadenada_encontrado_iou05.csv'),
]

def load(nb, pat):
    hits = sorted(glob.glob(f'{IN}/**/{nb}/{pat}', recursive=True))
    if not hits: return None
    if '*' in pat:
        parts = []
        for h in hits:
            m = re.search(r'matriz_icdas_(.+)\.csv', h); d = pd.read_csv(h)
            d.insert(0, 'model', m.group(1) if m else os.path.basename(h)); parts.append(d)
        return pd.concat(parts, ignore_index=True)
    return pd.read_csv(hits[0])

sheets, idx, missing = {}, [], []
for sh, sec, desc, nb, pat in SPEC:
    d = load(nb, pat)
    if d is None: missing.append(f'{nb}/{pat}'); continue
    if sh == 'S2_GradCAM_central':
        g = d.groupby('model').centro.mean().rename('central_fraction_all')
        g2 = d.pivot_table(index='model', columns='acierto', values='centro', aggfunc='mean')
        g2.columns = [f'central_fraction_{"correct" if bool(c) else "incorrect"}' for c in g2.columns]
        d = pd.concat([g, g2, d.groupby('model').size().rename('n_crops')], axis=1).reset_index().sort_values('central_fraction_all', ascending=False)
    if 'Unnamed: 0' in d.columns and d['Unnamed: 0'].dtype == object: d = d.rename(columns={'Unnamed: 0': 'row'})
    d = tidy(d); sheets[sh] = d
    idx.append(dict(sheet=sh, section=sec, description=desc, source=f'{nb} / {pat}', rows=len(d), columns=d.shape[1]))
print('hojas:', len(sheets), '| faltan:', missing or 'ninguno')
assert not missing, 'faltan archivos: revisar SPEC'

head = pd.DataFrame({'A': [
    'Additional file 3. Per-fold, per-class, per-tooth and subgroup results, and sensitivity analyses',
    'A two-stage deep learning pipeline for tooth-level caries screening in smartphone intraoral photographs',
    'All metrics were computed on pooled out-of-fold predictions of five-fold cross-validation grouped by child; 95% CI by bootstrap of children.',
    'Columns ending in _ci_low / _ci_high are 95% CI limits; _fold_mean / _fold_sd are the mean and SD across folds; p_holm is the Holm-adjusted p value.',
    'Column names follow the analysis code (Spanish terms translated where possible). Values are unrounded.',
    'ECE in S2_metrics_CI is computed on the pooled out-of-fold predictions; Table 3 of the article reports the mean ECE across folds (S2_by_fold).', '']})
with pd.ExcelWriter(OUT, engine='openpyxl') as xw:
    head.to_excel(xw, sheet_name='README', index=False, header=False)
    pd.DataFrame(idx).to_excel(xw, sheet_name='README', index=False, startrow=len(head))
    for sh, d in sheets.items(): d.to_excel(xw, sheet_name=sh, index=False)

wb = load_workbook(OUT)
for ws in wb.worksheets:
    if ws.title == 'README':
        ws['A1'].font = Font(bold=True, size=12)
        for c in ws[len(head) + 1]: c.font = Font(bold=True)
        for col, w in zip('ABCDEF', (24, 40, 110, 60, 8, 9)): ws.column_dimensions[col].width = w
        for row in ws.iter_rows(min_row=len(head) + 2):
            row[2].alignment = Alignment(wrap_text=True, vertical='top')
        continue
    ws.freeze_panes = 'B2'
    for c in ws[1]: c.font = Font(bold=True)
    for j, col in enumerate(ws.iter_cols(min_row=1, max_row=min(ws.max_row, 200)), 1):
        n = max(len(str(c.value)) if c.value is not None else 0 for c in col)
        ws.column_dimensions[get_column_letter(j)].width = min(max(8, n + 2), 45)
        for c in col[1:]:
            if isinstance(c.value, float): c.number_format = '0.0000'
wb.save(OUT)

chk = pd.ExcelFile(OUT)
print(os.path.getsize(OUT) // 1024, 'KB |', len(chk.sheet_names), 'hojas')
for sh in chk.sheet_names:
    d = chk.parse(sh); print(f'{sh:24s} {d.shape[0]:5d} x {d.shape[1]:3d} | {", ".join(map(str, d.columns[:8]))}')